A VPC (Virtual Private Cloud) is your own logically isolated network inside AWS. Every resource you deploy — EC2, RDS, Lambda in a VPC, ECS tasks — lives inside a VPC. Understanding how to design, secure, and connect VPCs is one of the most heavily tested areas of the Solutions Architect exam.

## VPC Basics

### What a VPC gives you
- Your own **private IP address space** (CIDR block)
- Full control over **subnets, route tables, and gateways**
- **Security** at the subnet level (NACLs) and instance level (security groups)
- Isolation from other AWS customers

### Scope
A VPC is **region-scoped** — it spans all Availability Zones in a region. Subnets within the VPC are AZ-scoped.

### CIDR blocks
When you create a VPC you assign an **IPv4 CIDR block** (e.g. `10.0.0.0/16`). The allowed range is `/16` (65,536 IPs) to `/28` (16 IPs). You can also add **secondary CIDR blocks** later if you need more address space.

IPv6 is also supported — AWS assigns a `/56` block; subnets get `/64`.

### Default VPC
Every AWS account comes with a **default VPC** in every region:
- CIDR `172.31.0.0/16`
- One public subnet per AZ, each with a `/20` block
- Internet Gateway already attached
- Instances launched into it get a public IP by default

The default VPC is convenient for quick testing but should not be used for production. Create custom VPCs with intentional public/private subnet design.

### Tenancy
- **Default**: your instances share physical hardware with other AWS customers (virtualised)
- **Dedicated**: all instances in the VPC run on dedicated hardware — higher cost

## Subnets

A subnet is a range of IP addresses within your VPC, confined to a **single AZ**.

### Public vs Private subnets
- **Public subnet**: has a route to an Internet Gateway → resources can be reached from the internet
- **Private subnet**: no route to an Internet Gateway → resources are not directly reachable from the internet

The distinction is entirely in the **route table**, not in any subnet property itself.

### Reserved IP addresses
AWS reserves **5 IPs** in every subnet — the first 4 and the last 1:

For `10.0.1.0/24`:
| Address | Reserved for |
|---|---|
| `10.0.1.0` | Network address |
| `10.0.1.1` | VPC router |
| `10.0.1.2` | DNS server |
| `10.0.1.3` | Future use |
| `10.0.1.255` | Broadcast (not used, but reserved) |

So a `/24` subnet gives you **251 usable IPs**, not 256.

> **Exam tip:** If asked "how many usable IPs in a /27 subnet?" — 32 total, minus 5 reserved = **27**.

### Auto-assign public IP
Each subnet has an **auto-assign public IPv4** setting. When enabled, instances launched into that subnet automatically get a public IP. This is on by default in public subnets of the default VPC, off by default in custom VPCs.

### Multi-AZ design pattern
A typical 3-tier VPC across 2 AZs:
```
VPC 10.0.0.0/16
├── Public  subnet A  10.0.1.0/24  (ALB, NAT Gateway)
├── Public  subnet B  10.0.2.0/24  (ALB, NAT Gateway)
├── Private subnet A  10.0.3.0/24  (App servers)
├── Private subnet B  10.0.4.0/24  (App servers)
├── Private subnet A  10.0.5.0/24  (Databases)
└── Private subnet B  10.0.6.0/24  (Databases)
```

## Internet Gateway & Route Tables

### Internet Gateway (IGW)
An IGW is a horizontally scaled, redundant VPC component that allows traffic between your VPC and the internet.

- **One IGW per VPC** (you cannot attach more than one)
- Attach it to the VPC, then add a route in the subnet's route table
- Performs **NAT for public IPs** — translates the instance's private IP to its public IP on outbound traffic and vice versa on inbound

For an instance to be reachable from the internet it needs **all three**:
1. A public IP (or Elastic IP)
2. A route to the IGW in its subnet's route table
3. A security group rule allowing the inbound traffic

### Route Tables
Every subnet is associated with a route table. The route table decides where traffic goes based on destination CIDR.

**Public subnet route table:**
| Destination | Target | Note |
|---|---|---|
| `10.0.0.0/16` | local | VPC-internal traffic stays local |
| `0.0.0.0/0` | igw-xxx | All other traffic → Internet Gateway |

**Private subnet route table:**
| Destination | Target | Note |
|---|---|---|
| `10.0.0.0/16` | local | VPC-internal only |
| `0.0.0.0/0` | nat-xxx | Outbound internet via NAT Gateway |

The `local` route is always present and cannot be deleted — it ensures all traffic within the VPC CIDR is routed locally.

**Longest prefix match**: if multiple routes match a destination, the most specific (longest prefix) wins.

## NAT Gateway

A NAT Gateway (Network Address Translation) allows instances in **private subnets** to initiate outbound connections to the internet (for software updates, API calls, etc.) without being directly reachable from the internet.

### How it works
```
Private EC2 → NAT Gateway (public subnet) → Internet Gateway → Internet
             ↑ translates private IP to NAT GW's Elastic IP
```

### Key characteristics
- **Managed by AWS** — no patching, no maintenance
- **AZ-scoped** — deploy one NAT Gateway per AZ for high availability
- Requires an **Elastic IP**
- Must be placed in a **public subnet**
- Supports up to **100 Gbps** bandwidth (auto-scales)
- **Stateful** — return traffic is automatically allowed

### High-availability pattern
```
AZ-a: Private subnet → NAT GW-a (public subnet AZ-a)
AZ-b: Private subnet → NAT GW-b (public subnet AZ-b)
```
If you use a single NAT Gateway in AZ-a and that AZ goes down, private instances in AZ-b lose internet access. Use one NAT Gateway per AZ.

### NAT Gateway vs NAT Instance
| | NAT Gateway | NAT Instance |
|---|---|---|
| Management | Fully managed | You manage EC2 |
| Availability | Highly available (AWS manages) | You configure HA |
| Bandwidth | Up to 100 Gbps | Instance type dependent |
| Security Groups | Cannot attach | Can attach |
| Cost | Higher | Lower (but operational overhead) |
| Bastion use | Cannot be used | Can double as bastion |

> **Exam tip:** Always prefer NAT Gateway over NAT Instance for new deployments. NAT Instance is legacy.

## Security Groups

Security groups are **instance-level** (more precisely: ENI-level) virtual firewalls.

### Key properties
- **Stateful**: if you allow inbound traffic on port 80, the return traffic is automatically allowed — no explicit outbound rule needed
- **Allow rules only** — you cannot write a deny rule in a security group
- Multiple security groups can be attached to one instance
- Rules can reference **another security group** (not just IP ranges) — allows you to say "allow traffic from the app-server SG" rather than hard-coding IPs

### Default security group
- **Inbound**: allows all traffic from the same security group (instances in the same SG can talk to each other)
- **Outbound**: allows all traffic to anywhere

### SG referencing pattern
```
ALB SG  →  App SG  →  DB SG

App SG inbound rule: allow port 8080 from ALB SG
DB SG  inbound rule: allow port 5432 from App SG
```
This is more maintainable than IP-based rules and automatically adapts as instances scale in/out.

## Network ACLs (NACLs)

NACLs are **subnet-level** firewalls. They are evaluated for traffic entering or leaving a subnet.

### Key properties
- **Stateless**: return traffic is NOT automatically allowed — you must explicitly allow both inbound and outbound rules
- Support **allow and deny** rules
- Rules are evaluated in **ascending rule number order** — first match wins
- Each subnet is associated with exactly one NACL (default NACL if not explicitly set)

### Default NACL
Allows **all inbound and all outbound** traffic. Created automatically with every VPC.

### Custom NACL
Denies **all traffic** until you add allow rules.

### Example: block a specific IP
| Rule | Type | Source | Allow/Deny |
|---|---|---|---|
| 100 | All traffic | 10.1.2.3/32 | **DENY** |
| 200 | All traffic | 0.0.0.0/0 | ALLOW |
| * | All traffic | 0.0.0.0/0 | DENY (implicit) |

Rule 100 matches first and blocks the specific IP; everything else hits rule 200 and is allowed.

### Ephemeral ports
When an instance responds to a request, it uses an **ephemeral (high) port** (1024–65535 for Linux). Because NACLs are stateless, you must allow outbound traffic on the ephemeral port range in addition to allowing the inbound request. Forgetting ephemeral port rules is a common NACL misconfiguration.

### SG vs NACL
| | Security Group | NACL |
|---|---|---|
| Level | Instance (ENI) | Subnet |
| State | Stateful | Stateless |
| Rules | Allow only | Allow + Deny |
| Evaluation | All rules evaluated | Rules evaluated in order (first match) |
| Default | Deny all inbound | Allow all (default NACL) |

## VPC Flow Logs

Flow Logs capture **metadata about IP traffic** flowing through your VPC — not the packet contents, just the headers: source IP, destination IP, port, protocol, bytes, packets, accept/reject status.

### Scope
You can enable flow logs at three levels:
- **VPC** — captures all traffic in the VPC
- **Subnet** — captures traffic for a specific subnet
- **ENI** — captures traffic for a specific network interface

### Destinations
- **S3** — for long-term storage and querying with Athena
- **CloudWatch Logs** — for real-time monitoring and alerting
- **Kinesis Data Firehose** — for streaming to third-party SIEM tools

### Use cases
- Diagnose why security group or NACL rules are rejecting traffic
- Audit which IPs are connecting to your resources
- Detect port scanning or unusual traffic patterns
- Capacity planning — identify high-traffic flows

Flow logs do **not** capture:
- Traffic to/from 169.254.169.254 (Instance Metadata Service)
- DHCP traffic
- DNS traffic to the VPC DNS resolver
- Traffic to the VPC router

## Bastion Hosts

A **bastion host** (jump server) is a small EC2 instance in a public subnet that you SSH into first, then SSH from there to private instances.

```
Your laptop → SSH → Bastion (public subnet) → SSH → App server (private subnet)
```

### Security hardening
- Restrict bastion's security group inbound SSH to **your IP only** (not 0.0.0.0/0)
- Use SSH agent forwarding so the private key never touches the bastion
- Use a small instance type — bastion doesn't need resources
- Consider replacing with **EC2 Instance Connect** or **AWS Systems Manager Session Manager** (no open SSH port required)

### Managed alternatives to bastions
| Option | How it works |
|---|---|
| **EC2 Instance Connect** | Browser or CLI SSH via AWS API — no open inbound SSH port needed |
| **Session Manager (SSM)** | Shell access via SSM agent — no SSH, no open ports, full audit trail in CloudTrail |

## DNS in a VPC

Every VPC has two DNS settings:

| Setting | What it enables |
|---|---|
| **enableDnsSupport** | Enables the VPC DNS resolver at `169.254.169.253` (or the base VPC CIDR + 2). Default: **on**. |
| **enableDnsHostnames** | Assigns public DNS hostnames (e.g. `ec2-1-2-3-4.compute-1.amazonaws.com`) to instances with public IPs. Default: **on** in default VPC, **off** in custom VPCs. |

Both must be enabled for Route 53 **private hosted zones** to resolve inside the VPC.

### Route 53 Resolver
The VPC DNS resolver handles:
- Resolving AWS service endpoints (e.g. `s3.amazonaws.com`)
- Resolving Route 53 private hosted zones
- Forwarding queries to on-premises DNS servers (via **Resolver Inbound/Outbound Endpoints**)

## Elastic IPs

An **Elastic IP (EIP)** is a static public IPv4 address you own in your account.

- Associates with an **ENI** (and therefore an instance)
- Persists when instance is stopped — unlike a regular public IP which is released on stop
- Can be **remapped** from one instance to another in minutes (used for fault tolerance)
- **Cost**: free while associated with a running instance; charged ~$0.005/hr when allocated but not associated (to discourage waste)
- Default limit: **5 EIPs per region** (can request increase)

> **Exam tip:** Use EIPs for NAT Gateways, VPN endpoints, or any service that external clients must reach at a fixed IP. Avoid attaching EIPs to individual app servers — use a load balancer instead.

## Building a VPC with boto3

In [ ]:
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

In [ ]:
# Create a VPC
vpc = ec2.create_vpc(
    CidrBlock='10.0.0.0/16',
    TagSpecifications=[{
        'ResourceType': 'vpc',
        'Tags': [{'Key': 'Name', 'Value': 'prod-vpc'}]
    }]
)
vpc_id = vpc['Vpc']['VpcId']

# Enable DNS hostnames (needed for private hosted zones)
ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={'Value': True})

print(f"VPC: {vpc_id}")

In [ ]:
# Create public and private subnets across two AZs
subnets = [
    ('10.0.1.0/24', 'us-east-1a', 'public-1a',  True),
    ('10.0.2.0/24', 'us-east-1b', 'public-1b',  True),
    ('10.0.3.0/24', 'us-east-1a', 'private-1a', False),
    ('10.0.4.0/24', 'us-east-1b', 'private-1b', False),
]

subnet_ids = {}
for cidr, az, name, public in subnets:
    s = ec2.create_subnet(
        VpcId=vpc_id, CidrBlock=cidr, AvailabilityZone=az,
        TagSpecifications=[{'ResourceType': 'subnet',
                            'Tags': [{'Key': 'Name', 'Value': name}]}]
    )
    sid = s['Subnet']['SubnetId']
    subnet_ids[name] = sid
    if public:
        ec2.modify_subnet_attribute(SubnetId=sid,
                                    MapPublicIpOnLaunch={'Value': True})
    print(f"{name}: {sid}")

In [ ]:
# Attach Internet Gateway
igw = ec2.create_internet_gateway()
igw_id = igw['InternetGateway']['InternetGatewayId']
ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)

# Create public route table with 0.0.0.0/0 → IGW
pub_rt = ec2.create_route_table(VpcId=vpc_id)
pub_rt_id = pub_rt['RouteTable']['RouteTableId']
ec2.create_route(RouteTableId=pub_rt_id,
                 DestinationCidrBlock='0.0.0.0/0',
                 GatewayId=igw_id)

# Associate public subnets with the public route table
for name in ['public-1a', 'public-1b']:
    ec2.associate_route_table(RouteTableId=pub_rt_id,
                              SubnetId=subnet_ids[name])

print(f"IGW {igw_id} attached, public route table {pub_rt_id} configured")

In [ ]:
# Create a NAT Gateway in each public subnet (one per AZ for HA)
# Each NAT Gateway needs an Elastic IP
for name, private_subnet in [('public-1a', 'private-1a'),
                               ('public-1b', 'private-1b')]:
    eip = ec2.allocate_address(Domain='vpc')
    nat = ec2.create_nat_gateway(
        SubnetId=subnet_ids[name],
        AllocationId=eip['AllocationId'],
        TagSpecifications=[{'ResourceType': 'natgateway',
                            'Tags': [{'Key': 'Name', 'Value': f'nat-{name}'}]}]
    )
    nat_id = nat['NatGateway']['NatGatewayId']

    # Create private route table: 0.0.0.0/0 → NAT Gateway
    priv_rt = ec2.create_route_table(VpcId=vpc_id)
    priv_rt_id = priv_rt['RouteTable']['RouteTableId']
    ec2.create_route(RouteTableId=priv_rt_id,
                     DestinationCidrBlock='0.0.0.0/0',
                     NatGatewayId=nat_id)
    ec2.associate_route_table(RouteTableId=priv_rt_id,
                              SubnetId=subnet_ids[private_subnet])

    print(f"NAT GW {nat_id} in {name}, private route table for {private_subnet}")

In [ ]:
# Enable VPC Flow Logs → CloudWatch Logs
ec2.create_flow_logs(
    ResourceIds=[vpc_id],
    ResourceType='VPC',
    TrafficType='ALL',          # ACCEPT, REJECT, or ALL
    LogDestinationType='cloud-watch-logs',
    LogGroupName='/aws/vpc/flow-logs',
    DeliverLogsPermissionArn='arn:aws:iam::123456789012:role/vpc-flow-logs-role'
)
print("Flow logs enabled → CloudWatch Logs")

## Summary

| Concept | Key Takeaway |
|---|---|
| VPC scope | Region-scoped; spans all AZs in the region |
| Subnet scope | AZ-scoped; public vs private determined by route table, not a flag |
| Reserved IPs | 5 per subnet (first 4 + last 1) |
| Internet Gateway | One per VPC; enables public internet access; performs NAT for public IPs |
| Route table | Longest prefix match; `local` route always present and cannot be deleted |
| NAT Gateway | Outbound-only internet for private subnets; deploy one per AZ for HA |
| NAT Instance | Legacy; avoid for new deployments |
| Security Group | Instance-level; stateful; allow only; supports SG referencing |
| NACL | Subnet-level; stateless; allow + deny; rules evaluated in order |
| Ephemeral ports | NACLs must allow 1024–65535 outbound for response traffic |
| VPC Flow Logs | IP traffic metadata; destinations: S3, CloudWatch Logs, Kinesis |
| Bastion host | Jump server for SSH to private instances; consider SSM Session Manager instead |
| Elastic IP | Static public IP; charged when unassociated; 5 per region default |
| DNS | enableDnsSupport + enableDnsHostnames both needed for private hosted zones |